# WITS — tariff rates (cut, on purpose)

**Status: deliberately cut. No connector, no data, and none planned.**
Owner: M2 Ekanayake. Reasoning: `ceynex-core/docs/DEFERRED.md`.

The World Bank's World Integrated Trade Solution would have supplied real
tariff schedules. The project plan named WITS as the **first thing to cut** if
the schedule slipped. It slipped by about two weeks, so the plan's own
instruction was followed.

This notebook is a decision record, not a scaffold. It shows exactly what the
cut costs, using the files that stand in for WITS today.

In [1]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

import _common as cx

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 5)

## 1. What was lost — exactly one thing

A "what if Sri Lanka loses GSP+" simulation needs a tariff rate to re-apply
once the preference is gone. That rate is now a documented constant instead of
a queried number.

Everything else is unaffected:

- Currency-shock questions never needed a tariff rate.
- User-supplied scenarios ("if the EU raised tariffs 10%") take the rate from
  the question itself.

In [2]:
elasticities = cx.CONFIG / "elasticities.yaml"
if elasticities.exists():
    print(elasticities.read_text())
else:
    print(f"not found: {elasticities}")

# Trade Economics agent — elasticity assumptions (SRS 3.1.5).
#
# These live in a documented config table rather than buried in code because
# SRS 3.1.5 requires the simulation to state its assumptions, and because a
# marker will ask where these numbers come from. Every entry needs a `source`.
#
# Populated properly on Day 7. Values below are PLACEHOLDERS carrying literature
# ranges, not fitted estimates — the agent must surface `basis` in its
# `assumptions` list so a reader knows which is which.

fx_pass_through:
  # Share of a depreciation that reaches USD export prices within one year.
  # <1.0 because contracts are priced in USD and repriced with a lag.
  agriculture:
    value: 0.6
    basis: literature_range
    source: "TBD Day 7 — cite the source used"
  apparel:
    value: 0.4
    basis: literature_range
    source: "TBD Day 7 — apparel is more contract-priced than agriculture"

export_demand_elasticity:
  # % change in export volume per 1% change in the buyer's landed pric

The `agreement_loss_mfn_tariff` block at the bottom is what WITS would have
replaced: **5.5% for agriculture, 9.5% for apparel.**

Two details worth knowing, both good answers to a marker's question:

**The constant is not applied blindly.** The agent uses it only after
`agreement_coverage()` has confirmed from the knowledge graph that a preference
actually exists for that product. Without that confirmation it reports that the
simulation cannot be completed, rather than applying a rate to a product no
agreement covers.

**Every entry carries `basis` and `source`, and the agent repeats them in its
assumptions on every run** — so an answer never presents an assumed rate as a
looked-up one.

> ⚠️ **Be honest about this if asked:** the `source` fields in this file still
> read `TBD`. The `basis` values are accurate — they really are
> `literature_range` and `assumption`, not fitted estimates — but the citations
> behind them were never filled in. The agent correctly reports the *basis*; it
> cannot report a source that is not there.

## 2. What still works without it

Agreement *coverage* — which deals apply to which products — is not WITS data.
It is maintained by hand in two committed CSVs, and it is what the knowledge
graph's `COVERED_BY` edges are built from.

In [3]:
agreements = cx.reference("trade_agreements.csv")
coverage = cx.reference("trade_agreement_coverage.csv")

print(f"trade_agreements.csv          {len(agreements)} rows")
display(agreements)
print(f"\ntrade_agreement_coverage.csv  {len(coverage)} rows")
display(coverage.head(15))

trade_agreements.csv          6 rows


,name,type,in_force_from,partners,source,verified
0,GSP+,unilateral_preference,2017-05-19,EU27,EU Regulation 978/2012 — Sri Lanka readmitted ...,unverified
1,UK DCTS,unilateral_preference,2023-06-19,GBR,UK Developing Countries Trading Scheme — succe...,unverified
2,ISFTA,bilateral_fta,2000-03-01,IND,India–Sri Lanka Free Trade Agreement,unverified
3,PSFTA,bilateral_fta,2005-06-12,PAK,Pakistan–Sri Lanka Free Trade Agreement,unverified
4,SAFTA,regional_fta,2006-01-01,BGD;BTN;IND;MDV;NPL;PAK;AFG,South Asian Free Trade Area,unverified
5,APTA,regional_pta,1976-06-17,BGD;CHN;IND;KOR;LAO;MNG,Asia-Pacific Trade Agreement,unverified



trade_agreement_coverage.csv  13 rows


,hs_code,agreement,from_year,to_year,note,verified
0,9,GSP+,2017,NaN,Coffee tea mate and spices — covers tea 0902 a...,unverified
1,15,GSP+,2017,NaN,Animal or vegetable fats and oils — covers coc...,unverified
2,40,GSP+,2017,NaN,Rubber and articles thereof,unverified
3,53,GSP+,2017,NaN,Other vegetable textile fibres — coir 5305,unverified
4,61,GSP+,2017,NaN,Knitted apparel — includes 6109 the SRS worked...,unverified
5,62,GSP+,2017,NaN,Woven apparel,unverified
6,61,UK DCTS,2023,NaN,Knitted apparel under the UK scheme,unverified
7,62,UK DCTS,2023,NaN,Woven apparel under the UK scheme,unverified
8,9,UK DCTS,2023,NaN,Coffee tea mate and spices under the UK scheme,unverified
9,9,ISFTA,2000,NaN,Subject to the ISFTA negative list — verify be...,unverified


> **Every row above is marked `unverified`.** Nobody has yet checked them
> against the official EU regulation or the ISFTA text. Under the team rule, no
> figure derived from them goes into a report until a human has. The graph
> stores that status on the node, so an answer that uses one can say so.
>
> This is a *different* gap from the WITS cut, and it is the more important of
> the two: a wrong coverage row changes which agreement applies, not just how
> precise the tariff is.

In [4]:
for name, frame in [("agreements", agreements), ("coverage", coverage)]:
    status_cols = [c for c in frame.columns if "verif" in c.lower() or "status" in c.lower()]
    if status_cols:
        print(f"{name}: {status_cols[0]} ->")
        print(frame[status_cols[0]].value_counts().to_string(), "\n")
    else:
        print(f"{name}: no verification-status column found; columns = {list(frame.columns)}\n")

agreements: verified ->
verified
unverified    6 

coverage: verified ->
verified
unverified    13 



## 3. What it would take to reinstate

WITS has a REST API and needs no key for the tariff endpoints CeyNex would use.
The work is roughly:

1. A connector fetching MFN and preferential rates per HS code and partner.
2. A `dim_tariff` table — these are rates, not trade flows, so they do not
   belong in `fact_trade`.
3. Replacing the `mfn_*` constants above with a lookup, keeping the constant as
   the documented fallback for codes WITS does not cover.

The honest reason it has not happened is time, not difficulty.